# Example 9.3 / 1: Write Decisions and Expressions

Companion notebook for **9.3 Writing Tables** in the AMPL lab Documentation, section "Write Decisions and Expressions". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Write Decisions and Expressions

The steel model reads `products.csv`, solves, and writes one row per product to `plan.csv`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # products
param rate {PROD} > 0;            # tons per hour
param profit {PROD};              # profit per ton
param market {PROD} >= 0;         # maximum sales, tons
param avail >= 0 default 40;      # hours available

var Make {p in PROD} >= 0, <= market[p];   # tons produced
maximize TotalProfit: sum {p in PROD} profit[p] * Make[p];
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;

table Products IN "amplcsv" "products.csv":
    PROD <- [product], rate, profit, market;
table Plan OUT "amplcsv" "plan.csv":
    {p in PROD} -> [product],
    Make[p] ~ make,
    profit[p] * Make[p] ~ contribution,
    round(Make[p].rc, 2) ~ reduced_cost;
table ProductPlan "amplcsv" "products.csv":
    PROD <- [product], rate IN, profit IN, market IN, Make OUT;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
with open("products.csv", "w") as f:
    f.write("product,rate,profit,market\n"
            "bands,200,25,6000\n"
            "coils,140,30,4000\n"
            "plate,160,29,3500\n")

In [5]:
# run.py
ampl.eval("load amplcsv.dll;")
ampl.read_table("Products")
ampl.solve(solver="highs", verbose=False)
if ampl.solve_result != "solved":
    raise RuntimeError(f"Not exporting: {ampl.solve_result}")
ampl.write_table("Plan")
print(open("plan.csv").read())

product,make,contribution,reduced_cost
bands,6000,150000,1.8
coils,0,0,-3.14
plate,1600,46400,0



### Pure AMPL: model.run

`{p in PROD} -> [product]` writes one row per member of `PROD` with the member in the `product`
column. Each data column is an expression in `p` with its heading after `~`: the variable value,
a computed contribution, and the reduced cost suffix `.rc` rounded to two decimals. Rename
suffixed and calculated columns this way: a heading such as `Make.rc` may be illegal in a
database even though the AMPL expression is valid. An indexed expression needs this
`{p in PROD} ->` form; with a plain `[product]` key AMPL rejects the declaration, because it
cannot tell which rows to write.

Pure AMPL, with the same `model.mod` and `products.csv`:

In [6]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [7]:
# data.py
with open("products.csv", "w") as f:
    f.write("product,rate,profit,market\n"
            "bands,200,25,6000\n"
            "coils,140,30,4000\n"
            "plate,160,29,3500\n")

In [8]:
%%ampl_eval
# model.run
load amplcsv.dll;
read table Products;
option solver highs;
option solver_msg 0;
solve;
if solve_result = "solved" then {
    write table Plan;
}
display Make;

Make [*] :=
bands  6000
coils     0
plate  1600
;



### Another way in amplpy

`plan.csv` then holds the same three rows as in the amplpy run.

**Write results next to the inputs.** Column directions let one declaration read some columns of a file and write others. With the
same model, `ProductPlan` reads `rate`, `profit` and `market` from `products.csv` and writes
`Make` into a new column of the same file:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
with open("products.csv", "w") as f:
    f.write("product,rate,profit,market\n"
            "bands,200,25,6000\n"
            "coils,140,30,4000\n"
            "plate,160,29,3500\n")

In [11]:
# run.py
ampl.eval("load amplcsv.dll;")
ampl.read_table("ProductPlan")
ampl.solve(solver="highs", verbose=False)
ampl.write_table("ProductPlan")
print(open("products.csv").read())

product,rate,profit,market,Make
bands,200,25,6000,6000
coils,140,30,4000,0
plate,160,29,3500,1600

